In [ ]:
SCAN_SET_ID = "ksr216_2026apr_repeats_01"

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("dark_background")

In [ ]:
from pprint import pprint

from rsoxs_unified_workup.config import load_scan_set
from rsoxs_unified_workup.loading import (
    build_base_description,
    scan_ids_by_polarization,
)

configuration = load_scan_set(SCAN_SET_ID)

print(build_base_description(configuration["sample"]))
pprint(scan_ids_by_polarization(configuration))

In [ ]:
import httpx
import PyHyperScattering as phs
from tiled.client import from_uri

catalog = from_uri(
    "https://tiled.nsls2.bnl.gov",
    timeout=httpx.Timeout(None),
)

loader = phs.load.SST1RSoXSDB(
    corr_mode="none",
    catalog=catalog["rsoxs"]["raw"],
)

In [ ]:
from rsoxs_unified_workup.loading import load_scans

scan = load_scans(loader, configuration)

for polarization, scan_data in scan.items():
    print(
        f"Scan {scan_data['scan_id']}: "
        f"configured={polarization}°, "
        f"measured={scan_data['data'].attrs['polarization']}°"
    )

In [ ]:
from rsoxs_unified_workup.plotting import plot_raw_frames

plot_raw_frames(scan)

In [ ]:
from rsoxs_unified_workup.darks import load_dark_frames

scan = load_dark_frames(loader, scan)

In [ ]:
from rsoxs_unified_workup.plotting import (
    plot_dark_histograms,
)

dark_upper_percentile = configuration[
    "processing"
]["darks"]["upper_percentile"]

dark_cutoffs = plot_dark_histograms(
    scan,
    upper_percentile=dark_upper_percentile,
)

{
    polarization: round(float(cutoff), 2)
    for polarization, cutoff in dark_cutoffs.items()
}

In [ ]:
from rsoxs_unified_workup.darks import build_dark_masks

dark_masks, dark_mask_summaries = build_dark_masks(
    scan,
    dark_cutoffs,
)

for polarization, summary in dark_mask_summaries.items():
    print(
        f"Scan {summary['scan_id']}, "
        f"polarization={polarization}°: "
        f"cutoff={summary['cutoff']:.2f}, "
        f"masked pixels={summary['masked_pixels']:,} "
        f"({summary['masked_percent']:.4f}%)"
    )

In [ ]:
from rsoxs_unified_workup.plotting import plot_dark_masks

plot_dark_masks(scan, dark_masks)